# 02. 상권 구조 프로필 생성

2024Q1~2025Q4 최근 8개 분기를 대상으로 area master의 1,650개 상권을 모두 유지한 `quarter × area_code` 구조 피처 패널을 생성한다. 추정매출은 사용하지 않는다.

API가 0값 상권을 행으로 제공하지 않는 블록은 숫자 0으로 채우되 `<dataset>_observed` 플래그를 남긴다. 점포 비율은 20개 점포 상당의 서울시 전체 prior로 smoothing하며, 극단값은 0.5%/99.5% 분위수로 clipping한다.

In [ ]:
from pathlib import Path
import sys
import pandas as pd

PROJECT_ROOT = Path.cwd().parent if Path.cwd().name == 'notebooks' else Path.cwd()
sys.path.insert(0, str(PROJECT_ROOT))

from src.features.area_profile import MODEL_EXCLUDE, build_area_profile
PROJECT_ROOT

In [ ]:
result = build_area_profile(project_root=PROJECT_ROOT)
profile = result.profile
feature_dictionary = result.feature_dictionary
validation = result.validation
merge_report = result.merge_report

In [ ]:
summary = pd.Series({
    'rows': len(profile),
    'areas': profile['area_code'].nunique(),
    'quarters': profile['quarter'].nunique(),
    'quarter_min': profile['quarter'].min(),
    'quarter_max': profile['quarter'].max(),
    'columns': len(profile.columns),
    'model_features': int(feature_dictionary['role'].eq('feature').sum()),
    'key_duplicate_rows': int(profile.duplicated(['quarter', 'area_code'], keep=False).sum()),
    'missing_cells': int(profile.isna().sum().sum()),
    'mean_data_reliability': profile['data_reliability'].mean(),
})
summary

## 블록별 조인 결과

조인 전후 행 수가 동일해야 하며, 미관측 키는 원본 행을 제거하지 않고 관측 플래그와 0 채움 정책으로 보존한다.

In [ ]:
merge_report[['dataset', 'source_rows', 'aggregated_rows', 'base_rows_before', 'rows_after_merge', 'matched_rows', 'missing_rows', 'join_rate', 'fill_policy']]

## 검증 및 분포 요약

2023→2024 호환성 경고는 확인했으며 이 결과에는 20241 이후 데이터만 포함한다. 상수 컬럼 경고에는 전 키가 관측된 데이터셋의 observed 플래그가 포함될 수 있다.

In [ ]:
validation.loc[validation['record_type'].eq('global_check'), ['name', 'value', 'status', 'details']]

In [ ]:
validation.loc[validation['record_type'].eq('feature_summary')].sort_values(['status', 'missing_rate'], ascending=[False, False]).head(25)

In [ ]:
profile[['quarter', 'area_code', 'area_name', 'observation_count', 'observed_block_count', 'data_reliability']].head()

산출물:

- `data/processed/area_profile.parquet`
- `outputs/tables/area_profile_feature_dictionary.csv`
- `outputs/tables/area_profile_validation.csv`
- `outputs/tables/area_profile_merge_report.csv`